In [3]:
# Set base path
base_path="../../"
%cd "$base_path"

/run/media/phyo-myat-oo/storage/Git_Repository/AI_Engineering_YKT/AIE-F-B2


In [5]:
# load text data

data_file_path = "data/myPOS/tag/mypos-ver.3.0.shuf.nopipe.txt"
!head "$data_file_path"

၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc
၂ဝ/num ရာစု/n မြန်မာ့/n သမိုင်း/n သန်းဝင်းလှိုင်/n ၊/punc ၂ဝဝ၉/num ခု/part ၊/punc မေ/n လ/n ၊/punc ကံကော်ဝတ်ရည်/n စာပေ/n ။/punc
ကျွန်တော်/pron မျက်မှန်/n တစ်/tn လက်/part လုပ်/v ချင်/part ပါ/p

In [ ]:

# load text data
test_file_path = "data/myPOS/tag/otest.1k.nopipe.txt"
!head "$data_file_path"


တစ်/tn ကိုက်/n ကို/ppm ဝမ်/n ခုနှစ်ထောင်/tn ပါ/part ။/punc
မနှစ်/n က/ppm သူ/pron ကျွန်မ/pron ကို/ppm သင်/v ပေး/part တယ်/ppm ။/punc
ကျွန်တော့်/pron ခုံ/n သွား/v ရှာ/v မလို့/part ။/punc
အတန်း/n စ/v တာ/part ကြာ/v ပြီ/ppm လား/part ။/punc
ဆေး/n နည်းနည်း/adv စား/v လိုက်/part ၊/punc သုံး/tn လေး/tn ရက်/n လောက်/part အနားယူ/v လိုက်/part ရင်/conj ပျောက်/v သွား/part မှာ/ppm ပါ/part ။/punc
အေးချမ်း/v မှု/part နဲ့/conj စည်းကမ်း/n ကို/ppm တည်မြဲ/v အောင်/part ထိန်းသိမ်း/v သည်/ppm ။/punc
ဇွန်း/n ကို/ppm လိုအပ်/v တယ်/ppm ။/punc
ဘွဲ့/n ရ/v ရင်/conj ဘာ/n လုပ်/v မ/part လို့/part လဲ/part ။/punc
ကျွန်တော်/pron ချောင်းဆိုး/v ခြင်း/part အတွက်/ppm တစ်/tn ခု/part ခု/part လို/v ချင်/part တယ်/ppm ။/punc
အသီးအနှံ/n တို့/part မှ/ppm လွဲ/v လျှင်/conj လူ/n တို့/part ၏/ppm အဓိက/n အစားအစာ/n မှာ/ppm ငါး/n ဖြစ်/v သည်/ppm ။/punc


In [ ]:
# prepare data 


import sys
import os

# Input and Output paths
INPUT_FILE = data_file_path
TRAIN_OUTPUT = "data/myPOS/tag/train.crfsuite"
TEST_OUTPUT = "data/myPOS/tag/test.crfsuite"
SPLIT_RATIO = 0.8  # 80% train, 20% test

def is_myanmar_or_arabic_digit(text):
    """Check if the text consists of Myanmar digits (၀-၉) or Arabic digits (0-9)."""
    myanmar_digits = set("၀၁၂၃၄၅၆၇၈၉0123456789")
    return len(text) > 0 and all(char in myanmar_digits for char in text)

def is_punctuation(text):
    """Check if the text is a punctuation mark."""
    punc_chars = set("။၊()\\_'\"")
    return text in punc_chars

def extract_word_features(sentence, i):
    """
    Extract features for word at position i in the given sentence list [(word, tag), ...].
    """
    word, tag = sentence[i]
    
    # Target label (POS Tag) MUST be the FIRST element for CRFSuite
    features = [
        tag,
        f"w[0]={word}",
        f"pref1={word[:1]}",
        f"suff1={word[-1:]}",
        f"suff2={word[-2:] if len(word) >= 2 else word}",
        f"is_digit={is_myanmar_or_arabic_digit(word)}",
        f"is_punc={is_punctuation(word)}"
    ]
    
    # Previous word feature (w[-1])
    if i > 0:
        prev_word, _ = sentence[i - 1]
        features.append(f"w[-1]={prev_word}")
    else:
        features.append("__BOS__")
        
    # Next word feature (w[1])
    if i < len(sentence) - 1:
        next_word, _ = sentence[i + 1]
        features.append(f"w[1]={next_word}")
    else:
        features.append("__EOS__")
        
    return features

def parse_line_to_sentence(line):
    """
    Parse a single raw line into a list of (word, tag) tuples.
    Example line: 'ဒီ/adj ဆေး/n က/ppm'
    Output: [('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm')]
    """
    line = line.strip()
    if not line:
        return []
    
    sentence = []
    tokens = line.split()
    for token in tokens:
        if '/' in token:
            # Split from the rightmost '/' to handle words that might contain '/'
            word, tag = token.rsplit('/', 1)
            sentence.append((word, tag))
    return sentence

def train_data():
    print(f"Loading raw dataset from {INPUT_FILE}...")
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found!")
        sys.exit(1)
        
    sentences = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            sent = parse_line_to_sentence(line)
            if sent:
                sentences.append(sent)
                
    print(f"Total sentences loaded: {len(sentences)}")
    
    # Train / Test split
    split_index = int(len(sentences) * SPLIT_RATIO)
    train_sentences = sentences[:split_index]
    test_sentences = sentences[split_index:]
    
    print(f"Training set: {len(train_sentences)} sentences")
    print(f"Testing set:  {len(test_sentences)} sentences")
    
    # Write Training Data
    print(f"Writing CRFSuite training features to {TRAIN_OUTPUT}...")
    with open(TRAIN_OUTPUT, "w", encoding="utf-8") as f:
        for sent in train_sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    # Write Testing Data
    print(f"Writing CRFSuite testing features to {TEST_OUTPUT}...")
    with open(TEST_OUTPUT, "w", encoding="utf-8") as f:
        for sent in test_sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    print("Data preparation complete!")

train_data()


Loading raw dataset from data/myPOS/tag/mypos-ver.3.0.shuf.nopipe.txt...
Total sentences loaded: 43196
Training set: 34556 sentences
Testing set:  8640 sentences
Writing CRFSuite training features to data/myPOS/tag/train.crfsuite...
Writing CRFSuite testing features to data/myPOS/tag/test.crfsuite...
Data preparation complete!


In [42]:
!head -n 10 $TRAIN_OUTPUT

num	w[0]=၁၉၆၂	pref1=၁	suff1=၂	suff2=၆၂	is_digit=True	is_punc=False	__BOS__	w[1]=ခုနှစ်
n	w[0]=ခုနှစ်	pref1=ခ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	w[-1]=၁၉၆၂	w[1]=ခန့်မှန်း
v	w[0]=ခန့်မှန်း	pref1=ခ	suff1=း	suff2=်း	is_digit=False	is_punc=False	w[-1]=ခုနှစ်	w[1]=သန်းခေါင်စာရင်း
n	w[0]=သန်းခေါင်စာရင်း	pref1=သ	suff1=း	suff2=်း	is_digit=False	is_punc=False	w[-1]=ခန့်မှန်း	w[1]=အရ
ppm	w[0]=အရ	pref1=အ	suff1=ရ	suff2=အရ	is_digit=False	is_punc=False	w[-1]=သန်းခေါင်စာရင်း	w[1]=လူဦးရေ
n	w[0]=လူဦးရေ	pref1=လ	suff1=ေ	suff2=ရေ	is_digit=False	is_punc=False	w[-1]=အရ	w[1]=၁၁၅၉၃၁
num	w[0]=၁၁၅၉၃၁	pref1=၁	suff1=၁	suff2=၃၁	is_digit=True	is_punc=False	w[-1]=လူဦးရေ	w[1]=ယောက်
part	w[0]=ယောက်	pref1=ယ	suff1=်	suff2=က်	is_digit=False	is_punc=False	w[-1]=၁၁၅၉၃၁	w[1]=ရှိ
v	w[0]=ရှိ	pref1=ရ	suff1=ိ	suff2=ှိ	is_digit=False	is_punc=False	w[-1]=ယောက်	w[1]=သည်
ppm	w[0]=သည်	pref1=သ	suff1=်	suff2=ည်	is_digit=False	is_punc=False	w[-1]=ရှိ	w[1]=။


In [44]:
!head -n 10 $TEST_OUTPUT

n	w[0]=လေကြောင်း	pref1=လ	suff1=း	suff2=်း	is_digit=False	is_punc=False	__BOS__	w[1]=စာပို့
v	w[0]=စာပို့	pref1=စ	suff1=့	suff2=ု့	is_digit=False	is_punc=False	w[-1]=လေကြောင်း	w[1]=စနစ်
n	w[0]=စနစ်	pref1=စ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	w[-1]=စာပို့	w[1]=ဖြင့်
ppm	w[0]=ဖြင့်	pref1=ဖ	suff1=့	suff2=့်	is_digit=False	is_punc=False	w[-1]=စနစ်	w[1]=ပို့
v	w[0]=ပို့	pref1=ပ	suff1=့	suff2=ု့	is_digit=False	is_punc=False	w[-1]=ဖြင့်	w[1]=ပါ
part	w[0]=ပါ	pref1=ပ	suff1=ါ	suff2=ပါ	is_digit=False	is_punc=False	w[-1]=ပို့	w[1]=။
punc	w[0]=။	pref1=။	suff1=။	suff2=။	is_digit=False	is_punc=True	w[-1]=ပါ	__EOS__

adv	w[0]=အခုတလော	pref1=အ	suff1=ာ	suff2=ော	is_digit=False	is_punc=False	__BOS__	w[1]=အပြင်ဘက်
n	w[0]=အပြင်ဘက်	pref1=အ	suff1=်	suff2=က်	is_digit=False	is_punc=False	w[-1]=အခုတလော	w[1]=ကို


In [47]:
# Good data format setting is working well. Next we train.
model_path="model/pos_model.crfsuite"
!time crfsuite learn -m $model_path  $TRAIN_OUTPUT

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-31T19:31:59Z

Reading the data set(s)
[1] data/myPOS/tag/train.crfsuite
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 34557
Seconds required: 3.834

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 34556
Number of items: 451891
Number of attributes: 65618
Number of labels: 15

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 111980
Seconds required: 1.338

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 999058.913901
Feature norm: 1.000000
Error norm: 193741.349574
Active features: 111980
Line search trials: 1
Line search step: 0.00

In [55]:
INPUT_FILE = test_file_path
OUTPUT_FILE = "data/myPOS/tag/otest.crfsuite"

def is_myanmar_or_arabic_digit(text):
    """Check if the text consists of Myanmar digits (၀-၉) or Arabic digits (0-9)."""
    myanmar_digits = set("၀၁၂၃၄၅၆၇၈၉0123456789")
    return len(text) > 0 and all(char in myanmar_digits for char in text)

def is_punctuation(text):
    """Check if the text is a punctuation mark."""
    punc_chars = set("။၊()\\_'\"")
    return text in punc_chars

def extract_word_features(sentence, i):
    """
    Extract features for word at position i in the given sentence list [(word, tag), ...].
    """
    word, tag = sentence[i]
    
    # Target label (POS Tag) MUST be the FIRST element for CRFSuite
    features = [
        tag,
        f"w[0]={word}",
        f"pref1={word[:1]}",
        f"suff1={word[-1:]}",
        f"suff2={word[-2:] if len(word) >= 2 else word}",
        f"is_digit={is_myanmar_or_arabic_digit(word)}",
        f"is_punc={is_punctuation(word)}"
    ]
    
    # Previous word feature (w[-1])
    if i > 0:
        prev_word, _ = sentence[i - 1]
        features.append(f"w[-1]={prev_word}")
    else:
        features.append("__BOS__")
        
    # Next word feature (w[1])
    if i < len(sentence) - 1:
        next_word, _ = sentence[i + 1]
        features.append(f"w[1]={next_word}")
    else:
        features.append("__EOS__")
        
    return features

def parse_line_to_sentence(line):
    """
    Parse a single raw line into a list of (word, tag) tuples.
    Example line: 'ဒီ/adj ဆေး/n က/ppm'
    Output: [('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm')]
    """
    line = line.strip()
    if not line:
        return []
    
    sentence = []
    tokens = line.split()
    for token in tokens:
        if '/' in token:
            # Split from the rightmost '/' to handle words that might contain '/'
            word, tag = token.rsplit('/', 1)
            sentence.append((word, tag))
    return sentence

def test_data():
    print(f"Loading raw test dataset from {INPUT_FILE}...")
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found!")
        sys.exit(1)
        
    sentences = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            sent = parse_line_to_sentence(line)
            if sent:
                sentences.append(sent)
                
    print(f"Total sentences loaded: {len(sentences)}")
    
    # Write 100% of sentences to output file
    print(f"Writing CRFSuite test features to {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for sent in sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    print(f"Data preparation complete! Processed {len(sentences)} sentences into {OUTPUT_FILE}.")

test_data()

Loading raw test dataset from data/myPOS/tag/otest.1k.nopipe.txt...
Total sentences loaded: 1000
Writing CRFSuite test features to data/myPOS/tag/otest.crfsuite...
Data preparation complete! Processed 1000 sentences into data/myPOS/tag/otest.crfsuite.


In [56]:
!crfsuite tag -m $model_path -qt $OUTPUT_FILE

Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (153, 153, 155) (1.0000, 0.9871, 0.9935)
    n: (2926, 3084, 3000) (0.9488, 0.9753, 0.9619)
    v: (1882, 1998, 2010) (0.9419, 0.9363, 0.9391)
    ppm: (2015, 2052, 2060) (0.9820, 0.9782, 0.9801)
    part: (3082, 3185, 3189) (0.9677, 0.9664, 0.9671)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    conj: (376, 437, 411) (0.8604, 0.9148, 0.8868)
    adj: (285, 331, 366) (0.8610, 0.7787, 0.8178)
    adv: (203, 225, 262) (0.9022, 0.7748, 0.8337)
    pron: (454, 469, 476) (0.9680, 0.9538, 0.9608)
    tn: (136, 140, 142) (0.9714, 0.9577, 0.9645)
    fw: (85, 88, 87) (0.9659, 0.9770, 0.9714)
    int: (22, 22, 25) (1.0000, 0.8800, 0.9362)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    abb: (11, 11, 12) (1.0000, 0.9167, 0.9565)
Macro-average precision, recall, F1: (0.957957, 0.933126, 0.944625)
Item accuracy: 12903 / 13468 (0.9580)
Instance accuracy: 638 / 1000 (0.6380)
Elapsed time: 0.025631 [sec] (39015.3 [

In [58]:
!crfsuite tag -m $model_path $OUTPUT_FILE> data/otest_predictions.txt